# 今晚去不去自习室？—— 决策树小案例

**场景**：根据当天的天气、是工作日还是周末、当天作业量、同学是否去自习室这四个因素，预测自己今晚会不会去自习室。

**为什么用决策树**：
1. 特征都是离散的（晴/阴/雨、多/中/少…），决策树天然适合；
2. 决策树可视化后就是一棵"如果…那么…"的树，可解释性强，能直接说出"为什么模型判断我会去"；
3. 数据量很小（40 条），复杂模型容易过拟合，决策树配合限制最大深度刚好。

**算法选择**：sklearn 默认实现的 **CART** 算法（用基尼系数划分），限制 `max_depth=3` 防止过拟合。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# 中文显示（Colab 上若不显示中文，下一格会自动安装中文字体）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 如果在 Colab 上中文显示成方块，运行这一格（本地有中文字体可跳过）
import matplotlib, os
if 'COLAB_GPU' in os.environ or not any('SimHei' in f.name or 'YaHei' in f.name for f in matplotlib.font_manager.fontManager.ttflist):
    !wget -q https://github.com/StellarCN/scp_zh/raw/master/fonts/SimHei.ttf -O /usr/share/fonts/truetype/SimHei.ttf 2>/dev/null
    matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/SimHei.ttf')
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
print('字体设置完成')

## 1. 构造数据集

手动模拟了 40 条"近期我每天的真实情况 + 当晚是否去了自习室"。
其中前 36 条覆盖了 3×2×3×2=36 种特征组合各一次，最后 4 条加入了一些"破坏规则"的噪声样本（更贴近真实情况：人并不是 100% 按规律行动的）。

In [ ]:
data = [
    ('晴','工作日','多','是','是'), ('晴','工作日','多','否','是'),
    ('晴','工作日','中','是','是'), ('晴','工作日','中','否','是'),
    ('晴','工作日','少','是','是'), ('晴','工作日','少','否','否'),
    ('晴','周末','多','是','是'),   ('晴','周末','多','否','是'),
    ('晴','周末','中','是','是'),   ('晴','周末','中','否','否'),
    ('晴','周末','少','是','否'),   ('晴','周末','少','否','否'),
    ('阴','工作日','多','是','是'), ('阴','工作日','多','否','是'),
    ('阴','工作日','中','是','是'), ('阴','工作日','中','否','是'),
    ('阴','工作日','少','是','是'), ('阴','工作日','少','否','否'),
    ('阴','周末','多','是','是'),   ('阴','周末','多','否','是'),
    ('阴','周末','中','是','是'),   ('阴','周末','中','否','否'),
    ('阴','周末','少','是','否'),   ('阴','周末','少','否','否'),
    ('雨','工作日','多','是','是'), ('雨','工作日','多','否','是'),
    ('雨','工作日','中','是','是'), ('雨','工作日','中','否','否'),
    ('雨','工作日','少','是','否'), ('雨','工作日','少','否','否'),
    ('雨','周末','多','是','是'),   ('雨','周末','多','否','否'),
    ('雨','周末','中','是','否'),   ('雨','周末','中','否','否'),
    ('雨','周末','少','是','否'),   ('雨','周末','少','否','否'),
    # 噪声样本：偶尔不按规律
    ('晴','工作日','中','否','否'),
    ('阴','周末','多','否','否'),
    ('雨','周末','多','是','否'),
    ('晴','周末','少','否','是'),
]
df = pd.DataFrame(data, columns=['天气','星期','作业量','同学去','去自习室'])
print(f'共 {len(df)} 条数据')
df.head(8)

In [ ]:
# 看一下正负样本比例
df['去自习室'].value_counts()

## 2. 特征编码

sklearn 的决策树要求输入是数值。用独热编码（one-hot）把类别特征展开成 0/1 列，目标列简化为 1=去 / 0=不去。

In [ ]:
X = pd.get_dummies(df[['天气','星期','作业量','同学去']])
y = (df['去自习室'] == '是').astype(int)
print('特征列：', list(X.columns))
X.head(3)

## 3. 划分训练 / 测试集，训练决策树

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
clf = DecisionTreeClassifier(
    criterion='gini',   # 也可换成 'entropy' 试试效果
    max_depth=3,        # 限制深度，防止过拟合
    random_state=42,
)
clf.fit(X_train, y_train)
print('模型已训练')

## 4. 评估模型

In [ ]:
pred = clf.predict(X_test)
print(f'测试集准确率：{accuracy_score(y_test, pred):.2%}\n')
print(classification_report(y_test, pred, target_names=['不去','去'], zero_division=0))

## 5. 把决策树画出来

决策树最大的优点就是"白盒"——能看到模型到底是按什么规则做判断的。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    clf,
    feature_names=X.columns.tolist(),
    class_names=['不去','去'],
    filled=True, rounded=True, fontsize=11, ax=ax,
)
plt.title('今晚去不去自习室 —— 决策树（max_depth=3）')
plt.show()

## 6. 特征重要性

可以直观看到哪些特征对决策影响最大。

In [ ]:
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=True)
imp = imp[imp > 0]
fig, ax = plt.subplots(figsize=(8, 4))
imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('特征重要性')
ax.set_xlabel('重要性')
plt.tight_layout()
plt.show()

## 7. 用几个新场景试试预测

In [ ]:
new_samples = pd.DataFrame([
    {'天气':'雨','星期':'工作日','作业量':'多','同学去':'否'},  # 作业多，按规律应该去
    {'天气':'晴','星期':'周末','作业量':'少','同学去':'否'},   # 周末没作业，多半不去
    {'天气':'阴','星期':'工作日','作业量':'中','同学去':'是'},  # 大概率去
])
new_X = pd.get_dummies(new_samples).reindex(columns=X.columns, fill_value=0)
preds = clf.predict(new_X)
proba = clf.predict_proba(new_X)[:, 1]
for s, p, pr in zip(new_samples.to_dict('records'), preds, proba):
    print(f"{s} → 预测：{'去' if p==1 else '不去'}（去的概率 {pr:.0%}）")

## 8. 小结

- 决策树把"经验"显式画成了一棵可读的判断树，对这种**小数据 + 离散特征 + 需要解释**的场景特别合适。
- `max_depth` 是最重要的防过拟合手段。本案例如果不限制深度，树会一路分裂到把每条噪声样本都"硬记下来"，测试集表现反而会变差——可以自己改成 `max_depth=None` 跑一下对比。
- 真正的项目里通常会用**随机森林 / 梯度提升树**（多棵决策树投票）来提升准确率，但单棵决策树永远是"快速理解数据"的好工具。